# 01 - Exploratory Data Analysis
Telco Customer Churn (7,043 customers). Run this notebook with the project's `venv` kernel from the `notebooks/` folder.

In [ ]:
import sys
sys.path.append("..")           # so we can import from src/
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.preprocess import load_and_clean
sns.set_theme(style="whitegrid")

## 1. Load and inspect the raw data

In [ ]:
raw = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(raw.shape)
raw.info()
raw.head()

In [ ]:
# TotalCharges is text because of 11 blank strings (customers with tenure = 0)
print((raw["TotalCharges"].str.strip() == "").sum(), "blank TotalCharges values")
print("Duplicates:", raw.duplicated().sum())

## 2. Clean (same function the model and app use)

In [ ]:
df = load_and_clean()
print(df.isnull().sum().sum(), "missing values")
df["Churn"].value_counts(normalize=True)

The target is imbalanced (~26.5% churn), so accuracy alone is misleading. We focus on recall, F1 and ROC-AUC.

## 3. Churn rate by category

In [ ]:
for col in ["Contract", "PaymentMethod", "InternetService", "TechSupport", "PaperlessBilling", "SeniorCitizen"]:
    print(df.groupby(col)["Churn"].mean().sort_values(ascending=False).round(3), "\n")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ["Contract", "PaymentMethod", "InternetService"]):
    df.groupby(col)["Churn"].mean().sort_values().plot.barh(ax=ax, color="#E4572E")
    ax.set_title(f"Churn rate by {col}"); ax.set_xlabel("Churn rate")
plt.tight_layout(); plt.show()

## 4. Numeric features

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(data=df, x="tenure", hue="Churn", multiple="stack", ax=axes[0])
sns.boxplot(data=df, x="Churn", y="MonthlyCharges", ax=axes[1])
sns.boxplot(data=df, x="Churn", y="TotalCharges", ax=axes[2])
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(df.select_dtypes("number").corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation (numeric features)"); plt.show()

## 5. Key insights (write yours here)
1. Month-to-month contracts churn far more than 1- or 2-year contracts.
2. Churn is concentrated in the first year of tenure.
3. Fiber optic customers and electronic-check payers churn more.
4. Customers without tech support / online security churn more.
5. Churners tend to have higher monthly charges.

Verify each of these against the numbers above and add the exact figures to the README.